In [7]:
import polars as pl
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram, cophenet
from scipy.spatial.distance import squareform
import matplotlib as plt

MH = pl.read_ipc("C:/Users/Ben/turbo-parakeet/tests/marginal_entropies.ipc")
JH = pl.read_ipc("C:/Users/Ben/turbo-parakeet/tests/pairwise_entropies.ipc")

In [8]:
joined = JH\
            .join(MH, how="left", left_on="col_a", right_on="col_name")\
            .rename({"entropy":"H_A_B", "entropy_right":"H_A"})\
            .join(MH, how="left", left_on="col_b", right_on="col_name")\
            .rename({"entropy":"H_B"})\
            .with_columns((2 - (pl.col("H_A")+pl.col("H_B"))/pl.col("H_A_B").round(8)).alias("VI"))\
            .with_columns((pl.col("H_A_B") - pl.col("H_A")).round(8).alias("H_B|A"))\
            .with_columns((pl.col("H_A_B") - pl.col("H_B")).round(8).alias("H_A|B"))\
            .sort(pl.col(["col_a", "col_b"]))

joined

col_a,col_b,H_A_B,H_A,H_B,VI,H_B|A,H_A|B
str,str,f64,f64,f64,f64,f64,f64
"""arr_high_unique""","""arr_sparse""",9.746125,9.65502,2.250699,0.778415,0.091104,7.495425
"""arr_skewed""","""arr_high_unique""",9.866358,4.42306,9.65502,0.573123,5.4432982,0.211338
"""arr_skewed""","""arr_sparse""",5.783752,4.42306,2.250699,0.846119,1.360692,3.533053
"""boolean_high_unique""","""arr_high_unique""",9.71847,1.18763,9.65502,0.884325,8.53084,0.06345
"""boolean_high_unique""","""arr_skewed""",5.557379,1.18763,4.42306,0.990407,4.369749,1.134318
…,…,…,…,…,…,…,…
"""uint32_sparse""","""categorical_skewed""",5.781162,2.250699,4.420857,0.845984,3.530463,1.360305
"""uint32_sparse""","""categorical_sparse""",3.913561,2.250699,2.250699,0.849795,1.662862,1.662862
"""uint32_sparse""","""list_high_unique""",9.746125,2.250699,9.676491,0.776212,7.495425,0.069634


# 1. Hierarchical clustering on Normalised Variation of Information

`VI` here is the *normalised* Variation of Information, `NVI(A,B) = VI(A,B) / H(A,B) = 2 − (H(A)+H(B)) / H(A,B)`.
It is a true metric on partitions, bounded in `[0, 1]`: **0 = the columns are injective relabelings of each other** (functionally equivalent), **1 = statistically independent**.

We feed the pairwise NVI matrix to agglomerative clustering with **average linkage** and draw the dendrogram with the root at the top. Branches are coloured by the (≤ 4) subtrees two levels below the root; **hover over any junction dot** to see the merge height and the full list of columns under that branch.

In [9]:
# --- NVI distance matrix -> average-linkage tree ---------------------------
labels = sorted(set(joined["col_a"]).union(joined["col_b"]))
n = len(labels)
idx = {c: i for i, c in enumerate(labels)}

D = np.zeros((n, n))
for a, b, vi in joined.select("col_a", "col_b", "VI").iter_rows():
    D[idx[a], idx[b]] = D[idx[b], idx[a]] = max(vi, 0.0)  # clip float-rounding negatives

condensed = squareform(D, checks=False)
Z = linkage(condensed, method="average")

# members[c] = column names under cluster id c (leaves are 0..n-1, merges n..2n-2)
members = {i: [labels[i]] for i in range(n)}
for k, (l, r, h, _) in enumerate(Z):
    members[n + k] = members[int(l)] + members[int(r)]

def children(c):
    return [int(Z[c - n][0]), int(Z[c - n][1])] if c >= n else []

def height(c):
    return Z[c - n][2] if c >= n else 0.0

def subtree_ids(root_id):
    out, stack = set(), [root_id]
    while stack:
        c = stack.pop()
        out.add(c)
        stack += children(c)
    return out

root = 2 * n - 2
level1 = children(root)                                     # the 2 branches off the root
level2 = [g for c in level1 for g in (children(c) or [c])]  # their children (<= 4 groups)
level2.sort(key=lambda g: -len(members[g]))  # biggest group takes palette slot 1 (best contrast)

coph_r, _ = cophenet(Z, condensed)
print(f"{n} columns | root merge NVI = {height(root):.3f} | cophenetic correlation = {coph_r:.3f}")

18 columns | root merge NVI = 0.952 | cophenetic correlation = 0.938


In [10]:
# --- interactive dendrogram (plotly) ----------------------------------------
import plotly.graph_objects as go

INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
PAL = ["#2a78d6", "#1baf7a", "#eda100", "#008300"]  # categorical slots 1-4 (validated)

dn = dendrogram(Z, labels=labels, no_plot=True)
leaf_x = {lbl: 5.0 + 10.0 * i for i, lbl in enumerate(dn["ivl"])}

# Map each plotted link back to its cluster id: scipy plots children before
# parents, so every link endpoint is either a leaf or an already-seen apex.
apex = {(round(x, 4), 0.0): idx[lbl] for lbl, x in leaf_x.items()}
merge_of = {frozenset((int(l), int(r))): n + k for k, (l, r, *_) in enumerate(Z)}
links = []  # (cluster_id, xs, ys)
for xs, ys in zip(dn["icoord"], dn["dcoord"]):
    c1 = apex[(round(xs[0], 4), round(ys[0], 9))]
    c2 = apex[(round(xs[3], 4), round(ys[3], 9))]
    cid = merge_of[frozenset((c1, c2))]
    apex[(round((xs[1] + xs[2]) / 2, 4), round(ys[1], 9))] = cid
    links.append((cid, xs, ys))

group_of = {}  # cluster id -> level-2 group index (top links stay unassigned)
for gi, g in enumerate(level2):
    for c in subtree_ids(g):
        group_of[c] = gi

fig = go.Figure()

# one polyline trace per colour (None-separated segments)
for gi, colour in [(None, MUTED)] + [(gi, PAL[gi % 4]) for gi in range(len(level2))]:
    xs_all, ys_all = [], []
    for cid, xs, ys in links:
        if group_of.get(cid) == gi:
            xs_all += list(xs) + [None]
            ys_all += list(ys) + [None]
    if not xs_all:
        continue
    name = "upper tree" if gi is None else f"group {gi + 1} ({len(members[level2[gi]])} cols)"
    fig.add_trace(go.Scatter(x=xs_all, y=ys_all, mode="lines", name=name,
                             line=dict(color=colour, width=2), hoverinfo="skip"))

# junction dots carry the hover: merge height + every column under the branch
fig.add_trace(go.Scatter(
    x=[(xs[1] + xs[2]) / 2 for _, xs, _ in links],
    y=[ys[1] for _, _, ys in links],
    mode="markers", showlegend=False,
    marker=dict(size=9, color=[PAL[group_of[cid] % 4] if cid in group_of else MUTED
                               for cid, _, _ in links],
                line=dict(color=SURFACE, width=2)),
    customdata=[f"{len(members[cid])} columns:<br>" + "<br>".join(sorted(members[cid]))
                for cid, _, _ in links],
    hovertemplate="merge NVI = %{y:.3f}<br>%{customdata}<extra></extra>",
))

fig.update_layout(
    title=dict(text="Functional distance between columns (NVI, average linkage)",
               font=dict(color=INK, size=16)),
    template="none", paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family='system-ui, "Segoe UI", sans-serif', color=MUTED, size=12),
    xaxis=dict(tickvals=[leaf_x[l] for l in dn["ivl"]], ticktext=dn["ivl"],
               tickangle=-45, showgrid=False, zeroline=False,
               tickfont=dict(color=INK, size=11)),
    yaxis=dict(title="NVI distance  (0 = equivalent, 1 = independent)",
               gridcolor=GRID, zeroline=False, range=[0, 1.02]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    height=560, margin=dict(l=60, r=20, t=90, b=140), hoverlabel=dict(align="left"),
)
fig.show()

In [11]:
# --- static fallback: the branches 1 and 2 levels below the root ------------
def show_groups(title, group_ids):
    print(title)
    for gi, g in enumerate(group_ids, 1):
        print(f"  group {gi}  (merges internally at NVI <= {height(g):.3f}, {len(members[g])} cols)")
        for col in sorted(members[g]):
            print(f"    {col}")
    print()

show_groups(f"Level 1 - the two branches off the root (split at NVI = {height(root):.3f}):", level1)
show_groups("Level 2 - their children (the coloured branches in the dendrogram):", level2)

Level 1 - the two branches off the root (split at NVI = 0.952):
  group 1  (merges internally at NVI <= 0.000, 1 cols)
    boolean_sparse
  group 2  (merges internally at NVI <= 0.938, 17 cols)
    arr_high_unique
    arr_skewed
    arr_sparse
    boolean_high_unique
    boolean_skewed
    categorical_high_unique
    categorical_skewed
    categorical_sparse
    float64_high_unique
    float64_skewed
    float64_sparse
    list_high_unique
    list_skewed
    list_sparse
    uint32_high_unique
    uint32_skewed
    uint32_sparse

Level 2 - their children (the coloured branches in the dendrogram):
  group 1  (merges internally at NVI <= 0.925, 16 cols)
    arr_high_unique
    arr_skewed
    arr_sparse
    boolean_skewed
    categorical_high_unique
    categorical_skewed
    categorical_sparse
    float64_high_unique
    float64_skewed
    float64_sparse
    list_high_unique
    list_skewed
    list_sparse
    uint32_high_unique
    uint32_skewed
    uint32_sparse
  group 2  (merges inte

# 2. Which columns determine which?

Conditional entropy is directional: `H(B|A) = H(A,B) − H(A)` is the uncertainty left in **B** once **A** is known. `H(B|A) ≈ 0` means A *functionally determines* B. To compare across columns of different cardinality we normalise it into the **uncertainty coefficient**

`U(B|A) = 1 − H(B|A) / H(B) = MI(A,B) / H(B) ∈ [0, 1]`

— the fraction of B's entropy explained by A (1 = A fully determines B, 0 = A tells us nothing about B).

We draw an edge **A → B** whenever `U(B|A) ≥ 0.95` (near-functional dependency) and rank columns by mean `U` over all other columns. **Caveat:** a high-cardinality, near-unique column determines *everything* trivially — a row key "explains" every other column. The ranking table therefore shows each column's own entropy `H`; a strong determinant with *low* `H` is the interesting kind (a compact column that still predicts others), while a determinant with `H ≈ log₂(n_rows)` is just behaving like a primary key.

In [12]:
# --- uncertainty coefficients for every ordered pair -------------------------
U_THRESH = 0.95  # near-functional dependency
EPS = 1e-9

directed = pl.concat([
    joined.select(src="col_a", dst="col_b", H_src="H_A", H_dst="H_B", H_cond=pl.col("H_B|A")),
    joined.select(src="col_b", dst="col_a", H_src="H_B", H_dst="H_A", H_cond=pl.col("H_A|B")),
]).with_columns(
    U=pl.when(pl.col("H_dst") > EPS)  # a constant column is determined by anything
      .then((1 - pl.col("H_cond") / pl.col("H_dst")).clip(0.0, 1.0))
      .otherwise(1.0)
)

ranking = (
    directed.group_by("src")
    .agg(
        H=pl.col("H_src").first(),
        mean_U=pl.col("U").mean(),
        n_determined=(pl.col("U") >= U_THRESH).sum(),
        determines=pl.col("dst").filter(pl.col("U") >= U_THRESH).sort(),
    )
    .sort("mean_U", descending=True)
)

top = ranking.row(0, named=True)
print(f"Most determining column: {top['src']!r}  (H = {top['H']:.2f} bits, "
      f"mean U = {top['mean_U']:.3f}, determines {top['n_determined']}/{n - 1} columns at U >= {U_THRESH})")
with pl.Config(fmt_str_lengths=120, tbl_rows=-1):
    display(ranking)

Most determining column: 'categorical_high_unique'  (H = 9.69 bits, mean U = 0.963, determines 17/17 columns at U >= 0.95)


src,H,mean_U,n_determined,determines
str,f64,f64,u32,list[str]
"""categorical_high_unique""",9.690664,0.963466,17,"[""arr_high_unique"", ""arr_skewed"", … ""uint32_sparse""]"
"""list_high_unique""",9.676491,0.960959,16,"[""arr_high_unique"", ""arr_skewed"", … ""uint32_sparse""]"
"""float64_high_unique""",9.662205,0.960182,15,"[""arr_high_unique"", ""arr_skewed"", … ""uint32_sparse""]"
"""arr_high_unique""",9.65502,0.957966,12,"[""arr_skewed"", ""arr_sparse"", … ""uint32_sparse""]"
"""uint32_high_unique""",9.640572,0.956128,10,"[""arr_high_unique"", ""arr_sparse"", … ""uint32_skewed""]"
"""arr_skewed""",4.42306,0.286169,0,[]
"""float64_skewed""",4.417344,0.285969,0,[]
"""categorical_skewed""",4.420857,0.285386,0,[]
"""list_skewed""",4.436005,0.283604,0,[]


In [13]:
# --- directed determination graph (A -> B iff U(B|A) >= threshold) ----------
# Mutual determination (U >= thresh both ways) means the columns are functionally
# equivalent, so we condense each equivalence clique (SCC) into one node and lay
# the resulting DAG out in tiers: determinants above, determined below.
import networkx as nx

edges = directed.filter(pl.col("U") >= U_THRESH)
G = nx.DiGraph()
G.add_nodes_from(labels)
for s, d, u in edges.select("src", "dst", "U").iter_rows():
    G.add_edge(s, d, U=u)

mutual = sorted({tuple(sorted((s, d))) for s, d in G.edges if G.has_edge(d, s)})
C = nx.condensation(G)  # SCCs -> DAG; C.nodes[c]["members"] = original columns
print(f"{G.number_of_edges()} determination edges at U >= {U_THRESH} | "
      f"{len(mutual)} mutual pairs -> {C.number_of_nodes()} nodes after condensing equivalents")

# tiered layout: topological generation = row; isolated columns drop to the bottom row
gens = list(nx.topological_generations(C))
layer = {c: gi for gi, gen in enumerate(gens) for c in gen}
for c in C.nodes:
    if C.degree(c) == 0:
        layer[c] = len(gens) - 1
pos, side = {}, {}
for gi in range(len(gens)):
    row = sorted((c for c in C.nodes if layer[c] == gi),
                 key=lambda c: min(C.nodes[c]["members"]))
    for i, c in enumerate(row):
        pos[c] = ((i + 1) / (len(row) + 1), -gi)
        side[c] = i % 2  # alternate label above/below so neighbours don't collide

fig = go.Figure()
fig.update_layout(annotations=[
    dict(x=pos[v][0], y=pos[v][1], ax=pos[u][0], ay=pos[u][1],
         axref="x", ayref="y", xref="x", yref="y", text="",
         showarrow=True, arrowhead=2, arrowsize=1.2, arrowwidth=1.5,
         arrowcolor="rgba(137,135,129,0.55)", standoff=24, startstandoff=16)
    for u, v in C.edges
])

seen_groups = set()
for c in C.nodes:
    mem = sorted(C.nodes[c]["members"])
    determined = sorted({t for m in mem for t in G.successors(m)} - set(mem))
    gi = group_of.get(idx[mem[0]])
    label = mem[0] if len(mem) == 1 else f"{mem[0]}<br>(+{len(mem) - 1} equivalent)"
    hover = ("<b>" + " ≡ ".join(mem) + "</b>"
             + f"<br>determines {len(determined)} column(s)"
             + ("<br>→ " + "<br>→ ".join(determined) if determined else ""))
    fig.add_trace(go.Scatter(
        x=[pos[c][0]], y=[pos[c][1]], mode="markers+text",
        marker=dict(size=12 + 1.5 * len(determined),
                    color=PAL[gi % 4] if gi is not None else MUTED,
                    line=dict(color=SURFACE, width=2)),
        text=[label], textfont=dict(color=INK, size=11),
        textposition="top center" if pos[c][1] == 0 else
                     ("bottom center" if side[c] else "top center"),
        name=f"group {gi + 1}" if gi is not None else "ungrouped",
        legendgroup=str(gi), showlegend=(gi not in seen_groups),
        customdata=[hover], hovertemplate="%{customdata}<extra></extra>",
    ))
    seen_groups.add(gi)

fig.update_layout(
    title=dict(text=f"Determination hierarchy: A → B where U(B|A) ≥ {U_THRESH}  "
                    f"(node size = columns determined)", font=dict(color=INK, size=16)),
    template="none", paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family='system-ui, "Segoe UI", sans-serif', color=MUTED, size=12),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[-len(gens) + 0.5, 0.6]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    height=520, margin=dict(l=20, r=20, t=90, b=30), hoverlabel=dict(align="left"),
)
fig.show()

70 determination edges at U >= 0.95 | 10 mutual pairs -> 14 nodes after condensing equivalents
